# Streaming-feature ablation + spectral candidates

Goal: (1) measure which of the 30 streaming features (ported from the batch bank
into `submissions/lgb.ipynb`, holdout TS-AUC 0.6029) actually earn gain in the
*streaming* GBM, and (2) test whether new streaming-spectral features add signal
(trailing rFFT entropy/centroid vs a Welch reference spectrum).

Method: extract features on a log-spaced online-step subsample for training rows
and full-resolution for held-out series, then
- LightGBM gain ranking on the full feature set,
- leave-one-family-out TS-AUC (drop each family, retrain, measure delta),
- single-family-alone TS-AUC.

TS-AUC = time-stratified AUC (per online-step AUC weighted by n_pos*n_neg).

In [1]:
import math
from bisect import bisect_left, bisect_right
from collections import deque

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

PATH = "/Users/minqi/Documents/ADIA_Lab_Structural_Break_Challenge/"

## StreamState — 30 base feats + `spec_*` candidates

Verbatim port of the submission extractor, extended with a trailing-window spectral block. `FAMILIES` tags features for family-level ablation.

In [2]:

def _autocorr(x, lag):
    if len(x) <= lag:
        return 0.0
    a, b = x[:-lag], x[lag:]
    a = a - a.mean(); b = b - b.mean()
    d = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / d) if d > 0 else 0.0


def _fit_ar(x, p=5):
    if len(x) <= p + 5:
        return None, 1.0
    rows = np.lib.stride_tricks.sliding_window_view(x, p + 1)
    Y = rows[:, -1]
    Xd = np.column_stack([np.ones(len(rows)), rows[:, :-1][:, ::-1]])
    beta, *_ = np.linalg.lstsq(Xd, Y, rcond=None)
    resid = Y - Xd @ beta
    return beta, float(resid.var()) if len(resid) else 1.0


def _norm_psd(seg, win):
    x = (seg - seg.mean()) * win
    sp = np.fft.rfft(x)
    p = (sp.real ** 2 + sp.imag ** 2)[1:]  # drop DC
    s = p.sum()
    if s <= 0 or not np.isfinite(s):
        return None
    return p / s


def _spec_ec(p, freqs, logk):
    ent = -float((p * np.log(p + 1e-12)).sum()) * logk
    cen = float((freqs * p).sum())
    return ent, cen


class StreamState:
    VAR_MIN = 20
    SPEC_W = 128

    _BASE = (
        "n_online", "mean_shift_z", "last_z", "online_logvar", "online_std",
        "online_skew", "online_kurt", "cumsum_range", "cusum_max",
        "tail2_frac", "tail3_frac", "jump_freq", "signrun_freq",
        "ks_stat", "rank_dev", "ac1_online", "ac1_shift", "ar_resid_logratio",
        "ref_log_n", "ref_skew", "ref_kurt", "ref_ac1",
    )
    _SPEC = ("spec_entropy", "spec_centroid", "spec_entropy_shift", "spec_centroid_shift")

    def __init__(self, bins=16, roll_windows=(50, 200),
                 ewma_alphas=(0.05, 0.2), ar_p=5, spectral=True):
        self.bins = bins; self.rw = tuple(roll_windows)
        self.alphas = tuple(ewma_alphas); self.p = ar_p; self.spectral = spectral
        self.feature_names = list(self._BASE)
        for a in self.alphas:
            self.feature_names += ["ewma%.2f_z" % a, "ewma%.2f_logvar" % a]
        for w in self.rw:
            self.feature_names += ["roll%d_meanshift" % w, "roll%d_logvar" % w]
        self._ew_off = len(self._BASE)
        self._roll_off = self._ew_off + 2 * len(self.alphas)
        self._spec_off = len(self.feature_names)
        if self.spectral:
            self.feature_names += list(self._SPEC)
        self.ncol = len(self.feature_names)
        self._out = np.zeros(self.ncol, dtype=np.float64)
        self._bin_targets = np.linspace(1.0 / self.bins, 1.0, self.bins)
        W = self.SPEC_W
        self._spec_win = np.hanning(W)
        self._spec_freqs = np.arange(1, W // 2 + 1, dtype=np.float64) / (W // 2)
        self._spec_logk = 1.0 / math.log(len(self._spec_freqs))
        self.FAMILIES = {
            "moments": ["n_online", "mean_shift_z", "last_z", "online_logvar",
                        "online_std", "online_skew", "online_kurt"],
            "changepoint": ["cumsum_range", "cusum_max", "ks_stat", "rank_dev"],
            "tail_jump": ["tail2_frac", "tail3_frac", "jump_freq", "signrun_freq"],
            "dependence": ["ac1_online", "ac1_shift", "ar_resid_logratio"],
            "ewma": ["ewma%.2f_z" % a for a in self.alphas]
                    + ["ewma%.2f_logvar" % a for a in self.alphas],
            "rolling": ["roll%d_meanshift" % w for w in self.rw]
                       + ["roll%d_logvar" % w for w in self.rw],
            "refctx": ["ref_log_n", "ref_skew", "ref_kurt", "ref_ac1"],
        }
        if self.spectral:
            self.FAMILIES["spectral"] = list(self._SPEC)

    def reset(self, reference):
        r = np.asarray(reference, dtype=np.float64)
        self.mu_h = float(r.mean()) if len(r) else 0.0
        self.sd_h = max(float(r.std(ddof=1)) if len(r) > 1 else 1.0, 1e-8)
        self.var_h = self.sd_h ** 2; self.ref_n = len(r)
        self._inv_refn = 1.0 / max(self.ref_n, 1)
        z = (r - self.mu_h) / self.sd_h if len(r) else r
        self.ref_skew = float((z ** 3).mean()) if len(r) else 0.0
        self.ref_kurt = float((z ** 4).mean() - 3.0) if len(r) else 0.0
        self.ref_ac1 = _autocorr(r, 1)
        self.ref_log_n = math.log(self.ref_n + 1.0)
        self.sorted_ref = np.sort(r).tolist() if len(r) else []
        edges = (np.quantile(r, np.linspace(0, 1, self.bins + 1)[1:-1])
                 if len(r) else np.zeros(self.bins - 1))
        self.edges = edges.tolist()
        self.ar, self.ar_var_h = _fit_ar(r, self.p)
        self.ref_spec_ent = 0.0; self.ref_spec_cen = 0.0
        if self.spectral and len(r) >= self.SPEC_W:
            W = self.SPEC_W; acc = np.zeros(W // 2); cnt = 0
            for s in range(0, len(r) - W + 1, W // 2):
                p = _norm_psd(r[s:s + W], self._spec_win)
                if p is not None:
                    acc += p; cnt += 1
            if cnt:
                pr = acc / acc.sum()
                self.ref_spec_ent, self.ref_spec_cen = _spec_ec(
                    pr, self._spec_freqs, self._spec_logk)
        self.n = 0
        self.mean = self.M2 = self.M3 = self.M4 = 0.0
        self.sx = self.sxx = self.sxy = 0.0
        self.cmax = self.cmin = self.cumsum = 0.0
        self.cusum_p = self.cusum_n = self.cusum_max = 0.0
        self.tail2 = self.tail3 = 0
        self.binc = np.zeros(self.bins); self.rank_sum = 0.0
        self.jumps = 0; self.signruns = 0; self.prev = None
        self.ar_resid_ss = 0.0; self.ar_n = 0
        self.arbuf = deque(maxlen=self.p)
        self.roll = {w: (deque(maxlen=w), [0.0, 0.0]) for w in self.rw}
        self.ew = {a: self.mu_h for a in self.alphas}
        self.ewv = {a: 0.0 for a in self.alphas}
        self.specbuf = deque(maxlen=self.SPEC_W)
        self._last_spec_ent = 0.0; self._last_spec_cen = 0.0
        return self

    def update(self, x):
        x = float(x); self.n += 1; n = self.n
        sd_h = self.sd_h; mu_h = self.mu_h; var_h = self.var_h
        d = x - self.mean; dn = d / n; dn2 = dn * dn
        term = d * dn * (n - 1)
        self.mean += dn
        self.M4 += term * dn2 * (n * n - 3 * n + 3) + 6 * dn2 * self.M2 - 4 * dn * self.M3
        self.M3 += term * dn * (n - 2) - 3 * dn * self.M2
        self.M2 += term
        var = self.M2 / (n - 1) if n > 1 else 0.0
        std = math.sqrt(var) if var > 0 else 0.0
        zx = (x - mu_h) / sd_h
        self.cumsum += zx
        if self.cumsum > self.cmax: self.cmax = self.cumsum
        if self.cumsum < self.cmin: self.cmin = self.cumsum
        cp = self.cusum_p + zx - 0.5; cn = self.cusum_n - zx - 0.5
        self.cusum_p = cp if cp > 0.0 else 0.0
        self.cusum_n = cn if cn > 0.0 else 0.0
        if self.cusum_p > self.cusum_max: self.cusum_max = self.cusum_p
        if self.cusum_n > self.cusum_max: self.cusum_max = self.cusum_n
        azx = zx if zx >= 0 else -zx
        if azx > 2.0: self.tail2 += 1
        if azx > 3.0: self.tail3 += 1
        if self.prev is not None:
            if abs(x - self.prev) > 2.0 * sd_h: self.jumps += 1
            if (x - mu_h) * (self.prev - mu_h) < 0: self.signruns += 1
            self.sxy += x * self.prev
        self.sx += x; self.sxx += x * x
        lo = bisect_left(self.sorted_ref, x); hi = bisect_right(self.sorted_ref, x)
        self.rank_sum += 0.5 * (lo + hi) * self._inv_refn
        self.binc[bisect_right(self.edges, x)] += 1
        if self.ar is not None and len(self.arbuf) == self.p:
            lag = np.array(self.arbuf, dtype=np.float64)[::-1]
            pred = self.ar[0] + self.ar[1:] @ lag
            r_ = x - pred; self.ar_resid_ss += r_ * r_; self.ar_n += 1
        self.arbuf.append(x)
        for a in self.alphas:
            m0 = self.ew[a]
            self.ew[a] = (1 - a) * m0 + a * x
            self.ewv[a] = (1 - a) * (self.ewv[a] + a * (x - m0) ** 2)
        for w, (buf, acc) in self.roll.items():
            if len(buf) == buf.maxlen:
                old = buf[0]; acc[0] -= old; acc[1] -= old * old
            buf.append(x); acc[0] += x; acc[1] += x * x
        if self.spectral:
            self.specbuf.append(x)
            if len(self.specbuf) == self.SPEC_W:
                p = _norm_psd(np.array(self.specbuf, dtype=np.float64), self._spec_win)
                if p is not None:
                    self._last_spec_ent, self._last_spec_cen = _spec_ec(
                        p, self._spec_freqs, self._spec_logk)
        self.prev = x

        out = self._out; se = sd_h / math.sqrt(n)
        emp = np.cumsum(self.binc) / n
        ks = float(np.max(np.abs(emp - self._bin_targets)))
        if n > 2 and var > 0:
            mx = self.sx / n
            cov = self.sxy / (n - 1) - mx * mx * n / (n - 1)
            ac1 = cov / var
        else:
            ac1 = 0.0
        vg = n >= self.VAR_MIN; sqrt_n = math.sqrt(n)
        out[0]=math.log1p(n); out[1]=(self.mean-mu_h)/max(se,1e-8); out[2]=zx
        out[3]=math.log((var+1e-8)/var_h) if vg else 0.0
        out[4]=std/sd_h
        out[5]=(self.M3/n)/(var**1.5+1e-8) if vg else 0.0
        out[6]=((self.M4/n)/(var**2+1e-8)-3.0) if vg else 0.0
        out[7]=(self.cmax-self.cmin)/sqrt_n; out[8]=self.cusum_max/sqrt_n
        out[9]=self.tail2/n; out[10]=self.tail3/n
        out[11]=self.jumps/n; out[12]=self.signruns/n
        out[13]=ks; out[14]=self.rank_sum/n-0.5; out[15]=ac1
        out[16]=ac1-self.ref_ac1
        out[17]=(math.log((self.ar_resid_ss/self.ar_n+1e-8)/(self.ar_var_h+1e-8))
                 if self.ar_n>=self.VAR_MIN else 0.0)
        out[18]=self.ref_log_n; out[19]=self.ref_skew
        out[20]=self.ref_kurt; out[21]=self.ref_ac1
        off=self._ew_off
        for ai,a in enumerate(self.alphas):
            ew_se=sd_h*math.sqrt(a/(2-a))
            out[off+2*ai]=(self.ew[a]-mu_h)/max(ew_se,1e-8)
            out[off+2*ai+1]=math.log((self.ewv[a]+1e-8)/var_h) if vg else 0.0
        off=self._roll_off
        for wi,w in enumerate(self.rw):
            buf,acc=self.roll[w]; L=len(buf)
            m=acc[0]/L; v=acc[1]/L-m*m
            if v<0.0: v=0.0
            out[off+2*wi]=(m-mu_h)/sd_h; out[off+2*wi+1]=math.log((v+1e-8)/var_h)
        if self.spectral:
            off=self._spec_off
            out[off]=self._last_spec_ent; out[off+1]=self._last_spec_cen
            out[off+2]=self._last_spec_ent-self.ref_spec_ent
            out[off+3]=self._last_spec_cen-self.ref_spec_cen
        return out


def _log_steps(L, m):
    if L <= m:
        return np.arange(L)
    return np.unique(np.round(np.expm1(np.linspace(0, np.log1p(L - 1), m))).astype(int))

_ss = StreamState()
COLS = _ss.feature_names
FAMILIES = _ss.FAMILIES
print(len(COLS), "features:", COLS)
print("families:", {k: len(v) for k, v in FAMILIES.items()})

34 features: ['n_online', 'mean_shift_z', 'last_z', 'online_logvar', 'online_std', 'online_skew', 'online_kurt', 'cumsum_range', 'cusum_max', 'tail2_frac', 'tail3_frac', 'jump_freq', 'signrun_freq', 'ks_stat', 'rank_dev', 'ac1_online', 'ac1_shift', 'ar_resid_logratio', 'ref_log_n', 'ref_skew', 'ref_kurt', 'ref_ac1', 'ewma0.05_z', 'ewma0.05_logvar', 'ewma0.20_z', 'ewma0.20_logvar', 'roll50_meanshift', 'roll50_logvar', 'roll200_meanshift', 'roll200_logvar', 'spec_entropy', 'spec_centroid', 'spec_entropy_shift', 'spec_centroid_shift']
families: {'moments': 7, 'changepoint': 4, 'tail_jump': 4, 'dependence': 3, 'ewma': 4, 'rolling': 4, 'refctx': 4, 'spectral': 4}


## Load data and split series

Deterministic split: first N_TRAIN ids for building rows, next N_HOLD ids held out for full-res eval.

In [3]:
X = pd.read_parquet(PATH + "X_train.parquet")
Yi = pd.read_parquet(PATH + "y_train_index.parquet")
tau_map = Yi["tau_index"].to_dict()

series = []  # (id, x_hist, x_online, tau)
for sid, sub in X.groupby(level="id", sort=False):
    vals = sub["value"].to_numpy(); per = sub["period"].to_numpy()
    t = int(tau_map[sid])
    series.append((sid, vals[per == 1], vals[per == 2], None if t == -1 else t))
print("total series:", len(series))

N_TRAIN = 6000   # build subsampled rows from these
N_HOLD  = 2000   # full-res eval on these
train_series = series[:N_TRAIN]
hold_series  = series[N_TRAIN:N_TRAIN + N_HOLD]
print("train", len(train_series), "hold", len(hold_series))

total series: 10000
train 6000 hold 2000


## Extract feature matrices

Training rows: log-spaced online-step subsample (~40/series). Holdout: full resolution, keeping (step, label) for time-stratified AUC. ~5-10 min in pure Python.

In [4]:
SAMPLES_PER = 40

def extract_train(series_list):
    st = StreamState()
    rows, labels, sids = [], [], []
    for sid, xh, xo, tau in series_list:
        st.reset(xh); L = len(xo)
        want = set(_log_steps(L, SAMPLES_PER).tolist())
        for k in range(L):
            out = st.update(xo[k])
            if k in want:
                rows.append(out.copy())
                labels.append(1 if (tau is not None and k >= tau) else 0)
                sids.append(sid)
    return (np.asarray(rows, np.float32), np.asarray(labels, np.int8), np.asarray(sids))

def extract_holdout(series_list):
    st = StreamState()
    rows, labels, steps = [], [], []
    for sid, xh, xo, tau in series_list:
        st.reset(xh); L = len(xo)
        for k in range(L):
            out = st.update(xo[k])
            rows.append(out.copy())
            labels.append(1 if (tau is not None and k >= tau) else 0)
            steps.append(k)
    return (np.asarray(rows, np.float32), np.asarray(labels, np.int8), np.asarray(steps, np.int32))

import time
t0 = time.time()
Xtr, ytr, sids = extract_train(train_series)
print("train rows", Xtr.shape, "in %.0fs" % (time.time()-t0))
t0 = time.time()
Xho, yho, steps = extract_holdout(hold_series)
print("holdout rows", Xho.shape, "in %.0fs" % (time.time()-t0))

train rows (197425, 34) in 144s


holdout rows (1000961, 34) in 37s


## TS-AUC helper + trainer

In [5]:
PARAMS = dict(objective="binary", n_estimators=500, learning_rate=0.05,
              num_leaves=63, min_child_samples=300, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
              n_jobs=-1, verbosity=-1)

def series_weights(sids):
    cmap = dict(zip(*np.unique(sids, return_counts=True)))
    w = np.array([1.0 / cmap[s] for s in sids]); return w / w.mean()

W_TR = series_weights(sids)

def ts_auc(y, score, steps):
    num = den = 0.0
    for t in np.unique(steps):
        m = steps == t; yv = y[m]
        npos = int(yv.sum()); nneg = len(yv) - npos
        if npos == 0 or nneg == 0: continue
        num += npos * nneg * roc_auc_score(yv, score[m]); den += npos * nneg
    return num / den if den else 0.5

def fit_eval(col_idx, seed=0):
    m = lgb.LGBMClassifier(random_state=seed, **PARAMS)
    m.fit(Xtr[:, col_idx], ytr, sample_weight=W_TR)
    sc = m.predict_proba(Xho[:, col_idx])[:, 1]
    return m, ts_auc(yho, sc, steps)

ALL = list(range(len(COLS)))
_, base_auc = fit_eval(ALL)
print("FULL feature set (%d feats) holdout TS-AUC = %.4f" % (len(COLS), base_auc))

/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


FULL feature set (34 feats) holdout TS-AUC = 0.5870


## 1. LightGBM gain ranking

In [6]:
m_full, _ = fit_eval(ALL)
gain = m_full.booster_.feature_importance(importance_type="gain")
rank = pd.DataFrame({"feature": COLS, "gain": gain}).sort_values("gain", ascending=False)
rank["gain_pct"] = 100 * rank["gain"] / rank["gain"].sum()
rank.reset_index(drop=True)

/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,feature,gain,gain_pct
0,n_online,155352.067518,21.798977
1,ref_kurt,78015.775205,10.947161
2,ref_log_n,66348.584221,9.310022
3,ref_skew,63554.581267,8.917968
4,ref_ac1,54341.309253,7.625164
5,spec_entropy_shift,44769.460091,6.282043
6,spec_centroid_shift,32272.621287,4.528489
7,ar_resid_logratio,21346.525785,2.995341
8,spec_entropy,15377.714453,2.157798
9,jump_freq,14472.662452,2.030802


## 2. Leave-one-family-out

Drop each family, retrain, measure TS-AUC delta vs full. Negative delta = family helps.

In [7]:
name2idx = {c: i for i, c in enumerate(COLS)}
loo = []
for fam, feats in FAMILIES.items():
    drop = set(feats); keep = [i for c, i in name2idx.items() if c not in drop]
    _, a = fit_eval(keep)
    loo.append((fam, len(feats), a, a - base_auc))
loo_df = pd.DataFrame(loo, columns=["family", "n_feat", "auc_without", "delta"]).sort_values("delta")
print("full TS-AUC = %.4f" % base_auc)
loo_df.reset_index(drop=True)

/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


full TS-AUC = 0.5870


,family,n_feat,auc_without,delta
0,dependence,3,0.578535,-0.008446
1,moments,7,0.582622,-0.004360
2,spectral,4,0.582911,-0.004071
3,rolling,4,0.586662,-0.000320
4,ewma,4,0.587234,0.000252
5,refctx,4,0.588071,0.001089
6,tail_jump,4,0.590981,0.003999
7,changepoint,4,0.591892,0.004910


## 3. Single-family-alone TS-AUC

In [8]:
solo = []
for fam, feats in FAMILIES.items():
    idx = [name2idx[c] for c in feats]
    _, a = fit_eval(idx)
    solo.append((fam, len(feats), a))
solo_df = pd.DataFrame(solo, columns=["family", "n_feat", "auc_alone"]).sort_values("auc_alone", ascending=False)
solo_df.reset_index(drop=True)

/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,family,n_feat,auc_alone
0,moments,7,0.545299
1,rolling,4,0.528032
2,ewma,4,0.525342
3,dependence,3,0.522852
4,tail_jump,4,0.518740
5,spectral,4,0.515194
6,refctx,4,0.508326
7,changepoint,4,0.496206


## 4. Spectral verdict

Does the `spectral` family earn its place? Compare full-with-spectral vs full-without.

In [9]:
spec_idx = [name2idx[c] for c in FAMILIES["spectral"]]
no_spec = [i for i in ALL if i not in set(spec_idx)]
_, auc_no_spec = fit_eval(no_spec)
print("full WITH spectral   : %.4f" % base_auc)
print("full WITHOUT spectral : %.4f" % auc_no_spec)
print("spectral delta        : %+.4f" % (base_auc - auc_no_spec))
print("spectral gain share   : %.1f%%" % rank.set_index("feature").loc[FAMILIES["spectral"], "gain_pct"].sum())

/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


full WITH spectral   : 0.5870
full WITHOUT spectral : 0.5829
spectral delta        : +0.0041
spectral gain share   : 13.1%
